Notebook desenvolvivo com base em: https://www.codegenes.net/blog/mvcnn-pytorch/

In [1]:
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from classes.mvcnn_dataset import MVCNNDataset
from classes.mvcnn_model import MVCNNModel
from torch.utils.data import DataLoader
from torchvision import transforms
from matplotlib import pyplot as plt
import torch.optim as optim
import seaborn as sns
import torch.nn as nn
import numpy as np
import random
import torch
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Função para retornar os arquivos existentes nos diretórios de cada indivíduo

In [2]:
def listar_diretorio(path):
    return [os.path.join(path, path_image) for path_image in os.listdir(path)]

Classes a serem classificadas

In [3]:
classes_shrimp = ["P. brasiliensis", "P. isabelae", "P. paulensis", "P. subtilis"]

class_to_idx = [
    i for i in range(0, len(classes_shrimp))
]
print(class_to_idx)

class_to_idx = {
    cls: i for i, cls in enumerate(classes_shrimp)
}

[0, 1, 2, 3]


Coletando diretórios das imagens dos indivíduos

In [4]:
from pathlib import Path

views_list_image_paths = []
labels_paths = []
for path in Path("dataset_mvcnn/").iterdir():

    if path.is_dir():
        label = path.name
        
        for class_dir in path.iterdir():

            if class_dir.iterdir():
                views_list_image_paths.append(listar_diretorio(class_dir))
                labels_paths.append(label)

print(len(views_list_image_paths))
print(len(labels_paths))

199
199


Transformers para treino, preparado para fazer data argumentation em memória

In [5]:
train_transform = transforms.Compose([ # Aplicando data argumentation na base de treino e aplicando transformadores para padronizar as imagens
    transforms.Resize((256,256)), # reduzindo tamanho das imagens

    transforms.RandomResizedCrop(
        224,
        scale=(0.8,1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5), # rotacionando imagens

    transforms.RandomRotation(15),# rotacionaod imagens

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize( # Normalizando dados
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

Transformer para teste

In [6]:
val_test_transform = transforms.Compose([
    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    views_list_image_paths,
    labels_paths,
    test_size=0.30,
    stratify=labels_paths,
    random_state=42
)

DataSet

In [8]:
train_dataset = MVCNNDataset(X_train, y_train, classes_shrimp, transform=train_transform)  

test_dataset = MVCNNDataset(X_test, y_test, classes_shrimp, transform=val_test_transform)

Dataloader

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

Função para treinamento do modelo

In [10]:
def train_model(model, train_loader, criterion, optimizer, num_epochs):

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    
    print(device)
    model = model.to(device)
    
    for epoch in range(num_epochs):

        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Acc: {epoch_acc:.2f}%"
        )

Função para avaliação do modelo

In [11]:
def evaluate_model(dataset, all_labels, all_preds, all_probs):
    class_names = dataset.classes

    print(f"\nAcurácia final: {accuracy_score(all_labels, all_preds)*100:.2f}%")
    print(f"F1-score macro: {f1_score(all_labels, all_preds, average='macro'):.2f}")
    print(f"F1-score weighted: {f1_score(all_labels, all_preds, average='weighted'):.2f}")

    print("\nClassification Report")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Classe prevista")
    plt.ylabel("Classe verdadeira")
    plt.title("Matriz de Confusão no Conjunto de Teste")
    plt.tight_layout()
    plt.show()

    # One-hot encoding dos rótulos verdadeiros
    y_true = label_binarize(all_labels, classes=list(range(4)))
    y_score = np.array(all_probs)

    # Plotar curva ROC para cada classe
    plt.figure(figsize=(10, 8))
    for i in range(4):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlabel('Taxa de Falsos Positivos (FPR)')
    plt.ylabel('Taxa de Verdadeiros Positivos (TPR)')
    plt.title('Curvas ROC por Classe')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

Função para teste do modelo

In [12]:
def test_model(model, test_loader):
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    
    model = model.to(device)
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, preds = torch.max(outputs, 1)

            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())


    evaluate_model(dataset=test_loader.dataset, all_labels=all_labels, all_preds=all_preds, all_probs=all_probs)

Preparando Modelo MVCNN (Utilizando resnet)

In [13]:
# Example usage
num_classes = 4
model_mvcnn = MVCNNModel(num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mvcnn.parameters(), lr=0.001)

c:\Users\USUARIO\Documents\camarao_rosa_mvcnn\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\USUARIO\Documents\camarao_rosa_mvcnn\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Treinando modelo MVCNN

In [ ]:
train_model(model_mvcnn, train_loader, criterion, optimizer, 50)

cuda
Epoch 01 | Loss: 0.7794 | Acc: 64.75%
Epoch 02 | Loss: 0.3238 | Acc: 89.93%
Epoch 03 | Loss: 0.2403 | Acc: 89.21%
Epoch 04 | Loss: 0.2368 | Acc: 91.37%
Epoch 05 | Loss: 0.0949 | Acc: 96.40%
Epoch 06 | Loss: 0.0625 | Acc: 97.84%
Epoch 07 | Loss: 0.1649 | Acc: 97.12%
Epoch 08 | Loss: 0.2477 | Acc: 93.53%
Epoch 09 | Loss: 0.0542 | Acc: 99.28%
Epoch 10 | Loss: 0.0642 | Acc: 97.84%
Epoch 11 | Loss: 0.0590 | Acc: 98.56%
Epoch 12 | Loss: 0.0664 | Acc: 97.84%
Epoch 13 | Loss: 0.1311 | Acc: 94.24%
Epoch 14 | Loss: 0.0696 | Acc: 97.84%
Epoch 15 | Loss: 0.0737 | Acc: 97.12%
Epoch 16 | Loss: 0.0242 | Acc: 100.00%
Epoch 17 | Loss: 0.0823 | Acc: 96.40%
Epoch 18 | Loss: 0.0952 | Acc: 94.96%
Epoch 19 | Loss: 0.2263 | Acc: 89.93%
Epoch 20 | Loss: 0.1012 | Acc: 94.96%


Testando modelo MVCNN

In [ ]:
test_model(model_mvcnn, test_loader)

['dataset_mvcnn\\P. brasiliensis\\PB_RN_02M\\PB_RN_02M_6_somito.JPG', 'dataset_mvcnn\\P. brasiliensis\\PB_RN_02M\\PB_RN_02M_carapaca.JPG', 'dataset_mvcnn\\P. brasiliensis\\PB_RN_02M\\PB_RN_02M_suco_adostral.JPG']
['dataset_mvcnn\\P. brasiliensis\\PB_RN_01M\\PB_RN_01M_6_somito.JPG', 'dataset_mvcnn\\P. brasiliensis\\PB_RN_01M\\PB_RN_01M_carapaca.JPG', 'dataset_mvcnn\\P. brasiliensis\\PB_RN_01M\\PB_RN_01M_suco_adostral.JPG']
['dataset_mvcnn\\P. brasiliensis\\PB_RN_04M\\PB_RN_04M_6_somito.JPG', 'dataset_mvcnn\\P. brasiliensis\\PB_RN_04M\\PB_RN_04M_carapaca.JPG', 'dataset_mvcnn\\P. brasiliensis\\PB_RN_04M\\PB_RN_04M_suco_adostral.JPG']
['dataset_mvcnn\\P. brasiliensis\\PB_RN_03M\\PB_RN_03M_6_somito.JPG', 'dataset_mvcnn\\P. brasiliensis\\PB_RN_03M\\PB_RN_03M_carapaca.JPG', 'dataset_mvcnn\\P. brasiliensis\\PB_RN_03M\\PB_RN_03M_suco_adostral.JPG']
Accuracy: 50.0%
Predicted class: 2
